# 01 — Limpeza e Consolidação

## Objetivo

Transformar a base bruta de clientes em uma **base limpa e consolidada**, com uma linha por
cliente, variáveis padronizadas e o rótulo `churn` (1 = cancelou nos 60 dias após a data de
corte, 0 = permaneceu). Essa base é a entrada de todas as etapas seguintes do projeto.

## Contexto

A operadora quer identificar, a partir da situação do cliente em **21/09/2026** (data de corte),
quem tem risco de cancelar o serviço até **20/11/2026**. Antes de qualquer modelo, os dados
precisam ser confiáveis: a base recebida tem clientes duplicados, categorias escritas de formas
diferentes, valores impossíveis e informações que ainda não estão no formato que um algoritmo
entende (datas e listas em vez de números).

Este notebook aplica as técnicas das aulas:
- **Aula 1** — entendimento do problema: granularidade, consolidação e criação do rótulo.
- **Aula 2** — qualidade dos dados: auditoria, padronização e tratamento de anomalias.

## Entrada e saída

| | Arquivo | Descrição |
| --- | --- | --- |
| **Entrada** | `data/raw/base_churn_vf2.csv` | Base original, 5.020 linhas |
| **Saída** | `data/processed/base_limpa.csv` | Base limpa, uma linha por cliente, com `churn` |

## Etapas

1. **Leitura** da base com os tipos corretos.
2. **Auditoria inicial** da qualidade (completude, validade, consistência e unicidade).
3. **Remoção de duplicatas.**
4. **Padronização das categorias** (de-para).
5. **Tratamento de valores sentinela** (−85, 9999 e 99 viram ausentes).
6. **Contagem de chamados** por cliente.
7. **Cálculo do tempo de casa** em meses.
8. **Criação do rótulo `churn`** e remoção de rótulos inconsistentes.
9. **Remoção de clientes imaturos** (menos de 2 meses de casa).
10. **Seleção das colunas finais**, auditoria final e exportação.

> **Importante:** aqui entram apenas correções que **não aprendem nada dos dados**.
> Imputação de ausentes, escalonamento e one-hot encoding dependem de estatísticas da base e
> ficam para o notebook 03, depois da divisão treino/teste, para evitar vazamento de dados.

In [1]:
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/previsao_de_evasao_de_clientes"):
        !git clone -q https://github.com/Gagdm/previsao_de_evasao_de_clientes.git /content/previsao_de_evasao_de_clientes
    %cd /content/previsao_de_evasao_de_clientes
    !git pull -q
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

import numpy as np
import pandas as pd
import config

/content/previsao_de_evasao_de_clientes


## 1. Leitura da base

Carregamos a base original sem nenhuma alteração. A mensalidade usa vírgula como separador
decimal, então informamos isso na leitura para que ela seja lida como número.
As colunas de data continuam como texto por enquanto: a auditoria precisa ver os formatos
originais, e a conversão é feita na etapa 7.

In [2]:
# Lê o CSV. (decimal "," faz o pandas entender o número com vírgula: 69,79 -> 69.79)
df_bruto = pd.read_csv(config.BASE_BRUTA, decimal=",")

# Printa a quantidade de linhas e colunas.
print(f"Linhas: {df_bruto.shape[0]} | Colunas: {df_bruto.shape[1]}")
# Mostra as 5 primeiras linhas para conferir se a leitura deu certo
df_bruto.head()

Linhas: 5020 | Colunas: 10


/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,id_cliente,data_aquisicao_cliente,mensalidade_rs,data_chamado,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,data_ocorrencia_churn
0,C0001,21/04/2024,67.96,"09/09/2026, 15/08/2026",0.0,Bienal,Boleto,0,0,NaN
1,C0002,21/02/2025,65.95,NaN,0.0,Mensal,Cartão,0,0,NaN
2,C0003,21/03/2025,100.88,28/08/2026,1.0,Anual,Boleto,1,1,NaN
3,C0004,21/03/2025,94.80,"06/08/2026, 28/07/2026",2.0,Mensal,Cartão,0,1,NaN
4,C0005,21/02/2022,53.68,05/07/2026,2.0,Anual,Pix,0,0,NaN


In [3]:
# Lista cada coluna com seu tipo e quantos valores não vazios ela tem. Primeiro diagnóstico da base.
df_bruto.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5020 entries, 0 to 5019
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_cliente              5020 non-null   object 
 1   data_aquisicao_cliente  5020 non-null   object 
 2   mensalidade_rs          4900 non-null   float64
 3   data_chamado            3967 non-null   object 
 4   atrasos_6m              4985 non-null   float64
 5   tipo_contrato           4994 non-null   object 
 6   forma_pagamento         5000 non-null   object 
 7   fibra_optica            5020 non-null   int64  
 8   debito_automatico       5020 non-null   int64  
 9   data_ocorrencia_churn   1789 non-null   object 
dtypes: float64(2), int64(2), object(6)
memory usage: 392.3+ KB


## 2. Auditoria inicial

Medimos a qualidade da base bruta nas quatro dimensões da Aula 2, sem alterar nada.
O resultado serve de comparação com a base limpa no final do notebook.

### 2.1 Completude — o dado está preenchido?

Consideramos só as colunas obrigatórias. `data_chamado` e `data_ocorrencia_churn` ficam de fora:
nelas, vazio tem significado (o cliente não abriu chamado ou não cancelou).

In [4]:
colunas_obrigatorias = ["id_cliente", "data_aquisicao_cliente", "mensalidade_rs", "atrasos_6m",
                        "tipo_contrato", "forma_pagamento", "fibra_optica", "debito_automatico"]

completude = pd.DataFrame({
    # Cria uma máscara com True onde a célula está vazia. Com .sum(), conta os vazios de cada coluna.
    "ausentes": df_bruto[colunas_obrigatorias].isna().sum(),
    # Dá a proporção de células preenchidas por coluna. O * 100 transforma em porcentagem.
    "% preenchido": (df_bruto[colunas_obrigatorias].notna().mean() * 100).round(2),
})

# Faz a média de todas as células juntas, não por coluna. É a completude geral.
pct_completude = df_bruto[colunas_obrigatorias].notna().values.mean() * 100
print(f"Completude geral: {pct_completude:.1f}%")
completude

Completude geral: 99.5%


,ausentes,% preenchido
id_cliente,0,100.00
data_aquisicao_cliente,0,100.00
mensalidade_rs,120,97.61
atrasos_6m,35,99.30
tipo_contrato,26,99.48
forma_pagamento,20,99.60
fibra_optica,0,100.00
debito_automatico,0,100.00


### 2.2 Validade — o valor respeita as regras do dicionário?

Cada coluna da tabela `violacoes` é uma regra; `True` indica que a linha quebra aquela regra.

In [5]:
canonicos_contrato = set(config.DE_PARA["tipo_contrato"].values())
canonicos_pagamento = set(config.DE_PARA["forma_pagamento"].values())
min_mensalidade, max_mensalidade = config.FAIXAS_VALIDAS["mensalidade_rs"]
min_atrasos, max_atrasos = config.FAIXAS_VALIDAS["atrasos_6m"]

violacoes = pd.DataFrame({
    "contrato_nao_canonico": df_bruto["tipo_contrato"].notna()
        & ~df_bruto["tipo_contrato"].isin(canonicos_contrato),
    "pagamento_nao_canonico": df_bruto["forma_pagamento"].notna()
        & ~df_bruto["forma_pagamento"].isin(canonicos_pagamento),
    "mensalidade_fora_da_faixa": df_bruto["mensalidade_rs"].notna()
        & ~df_bruto["mensalidade_rs"].between(min_mensalidade, max_mensalidade),
    "atrasos_fora_da_faixa": df_bruto["atrasos_6m"].notna()
        & ~df_bruto["atrasos_6m"].between(min_atrasos, max_atrasos),
    "binaria_invalida": ~df_bruto["fibra_optica"].isin([0, 1])
        | ~df_bruto["debito_automatico"].isin([0, 1]),
    "data_fora_do_formato": ~df_bruto["data_aquisicao_cliente"].str.match(r"^\d{2}/\d{2}/\d{4}$"),
})

linhas_invalidas = violacoes.any(axis=1)
pct_validade = (1 - linhas_invalidas.mean()) * 100
print(f"Linhas com alguma violação: {linhas_invalidas.sum()} | Validade: {pct_validade:.1f}%")
violacoes.sum()

Linhas com alguma violação: 203 | Validade: 96.0%


,0
contrato_nao_canonico,80
pagamento_nao_canonico,90
mensalidade_fora_da_faixa,12
atrasos_fora_da_faixa,8
binaria_invalida,0
data_fora_do_formato,15


In [6]:
print(df_bruto.loc[violacoes["contrato_nao_canonico"], "tipo_contrato"].map(repr).value_counts(), "\n")
print(df_bruto.loc[violacoes["pagamento_nao_canonico"], "forma_pagamento"].value_counts(), "\n")
print(df_bruto.loc[violacoes["mensalidade_fora_da_faixa"], "mensalidade_rs"].value_counts(), "\n")
print(df_bruto.loc[violacoes["atrasos_fora_da_faixa"], "atrasos_6m"].value_counts(), "\n")
print(df_bruto.loc[violacoes["data_fora_do_formato"], "data_aquisicao_cliente"].head())

tipo_contrato
'mensal'     40
'anual'      25
'bienal '    15
Name: count, dtype: int64 

forma_pagamento
cartao    50
boleto    25
PIX       15
Name: count, dtype: int64 

mensalidade_rs
-85.0      8
 9999.0    4
Name: count, dtype: int64 

atrasos_6m
99.0    8
Name: count, dtype: int64 

88     2024-08-21
386    2026-06-21
550    2022-09-21
700    2021-12-21
890    2024-12-21
Name: data_aquisicao_cliente, dtype: object


### 2.3 Consistência — as colunas concordam entre si?

Duas regras lógicas: o cliente não pode cancelar nem abrir chamado antes de ter sido adquirido.
As datas são convertidas só para esta verificação; `df_bruto` não é alterado.

In [7]:
aquisicao = pd.to_datetime(df_bruto["data_aquisicao_cliente"], format="mixed", dayfirst=True)
cancelamento = pd.to_datetime(df_bruto["data_ocorrencia_churn"], format="%d/%m/%Y")
churn_antes_da_aquisicao = cancelamento < aquisicao

chamados = pd.to_datetime(df_bruto["data_chamado"].str.split(", ").explode(), format="%d/%m/%Y")
chamado_antes_da_aquisicao = (
    (chamados < aquisicao.loc[chamados.index])
    .groupby(level=0).any()
    .reindex(df_bruto.index, fill_value=False)
)

inconsistencias = pd.DataFrame({
    "churn_antes_da_aquisicao": churn_antes_da_aquisicao,
    "chamado_antes_da_aquisicao": chamado_antes_da_aquisicao,
})

linhas_inconsistentes = inconsistencias.any(axis=1)
pct_consistencia = (1 - linhas_inconsistentes.mean()) * 100
print(f"Linhas inconsistentes: {linhas_inconsistentes.sum()} | Consistência: {pct_consistencia:.1f}%")
print(inconsistencias.sum(), "\n")
df_bruto.loc[churn_antes_da_aquisicao, ["id_cliente", "data_aquisicao_cliente", "data_ocorrencia_churn"]]

Linhas inconsistentes: 43 | Consistência: 99.1%
churn_antes_da_aquisicao      10
chamado_antes_da_aquisicao    33
dtype: int64 



,id_cliente,data_aquisicao_cliente,data_ocorrencia_churn
202,C0203,21/08/2024,15/01/2020
494,C0495,21/08/2025,15/01/2020
1356,C1357,21/12/2024,15/01/2020
1572,C1573,21/10/2021,15/01/2020
2026,C2027,21/12/2024,15/01/2020
2722,C2723,21/09/2024,15/01/2020
2999,C3000,21/03/2026,15/01/2020
3522,C3523,21/06/2026,15/01/2020
3644,C3645,21/05/2026,15/01/2020
4914,C4915,21/10/2025,15/01/2020


### 2.4 Unicidade — cada cliente aparece uma vez só?

In [8]:
ids_repetidos = df_bruto["id_cliente"].duplicated()
linhas_identicas = df_bruto.duplicated()

pct_unicidade = (1 - ids_repetidos.mean()) * 100
print(f"IDs repetidos: {ids_repetidos.sum()}")
print(f"Linhas idênticas a outra: {linhas_identicas.sum()}")
print(f"Repetidos com conteúdo diferente: {(ids_repetidos & ~linhas_identicas).sum()}")
print(f"Unicidade: {pct_unicidade:.1f}%")

IDs repetidos: 20
Linhas idênticas a outra: 20
Repetidos com conteúdo diferente: 0
Unicidade: 99.6%


### 2.5 Por que a mensalidade está faltando? (MCAR, MAR ou MNAR)

Se a ausência for igual em todos os grupos, ela parece aleatória (MCAR). Se quem não tem
mensalidade cancela mais, a própria ausência carrega informação (MNAR) e vale guardar um indicador.

In [9]:
mensalidade_ausente = df_bruto["mensalidade_rs"].isna()
cancelou = df_bruto["data_ocorrencia_churn"].notna()

print("% de mensalidade ausente por fibra óptica:")
print(mensalidade_ausente.groupby(df_bruto["fibra_optica"]).mean().mul(100).round(1), "\n")

print("% que cancelou, com e sem mensalidade:")
print(cancelou.groupby(mensalidade_ausente).mean().mul(100).round(1))

% de mensalidade ausente por fibra óptica:
fibra_optica
0    2.4
1    2.3
Name: mensalidade_rs, dtype: float64 

% que cancelou, com e sem mensalidade:
mensalidade_rs
False    35.5
True     40.0
Name: data_ocorrencia_churn, dtype: float64


### 2.6 Resumo da auditoria inicial

In [10]:
resumo_antes = pd.DataFrame({
    "dimensão": ["Completude", "Validade", "Consistência", "Unicidade"],
    "% conformidade": [pct_completude, pct_validade, pct_consistencia, pct_unicidade],
    "o que mede": ["células obrigatórias preenchidas",
                   "linhas sem valor fora das regras",
                   "linhas sem contradição entre colunas",
                   "linhas com id_cliente único"],
}).round(1)
resumo_antes

,dimensão,% conformidade,o que mede
0,Completude,99.5,células obrigatórias preenchidas
1,Validade,96.0,linhas sem valor fora das regras
2,Consistência,99.1,linhas sem contradição entre colunas
3,Unicidade,99.6,linhas com id_cliente único


## 3. Remoção de duplicatas

A auditoria mostrou 20 linhas idênticas a outras, sem nenhum caso de mesmo cliente com dados
diferentes. Basta manter uma cópia de cada.

A partir daqui trabalhamos em `df`, uma cópia de `df_bruto`, que fica intacta para a auditoria
final. Cada correção é registrada em `log_correcoes`.

In [11]:
df = df_bruto.copy()
log_correcoes = []

linhas_antes = len(df)
ids_duplicados = df.loc[df.duplicated(), "id_cliente"].tolist()

df = df.drop_duplicates().reset_index(drop=True)

removidas = linhas_antes - len(df)
log_correcoes.append({
    "etapa": "3. Duplicatas",
    "descricao": "linhas idênticas removidas",
    "linhas_afetadas": removidas,
})

print(f"Antes: {linhas_antes} | Depois: {len(df)} | Removidas: {removidas}")
print(f"id_cliente é único? {df['id_cliente'].is_unique}")
print(f"IDs que estavam duplicados: {ids_duplicados}")

Antes: 5020 | Depois: 5000 | Removidas: 20
id_cliente é único? True
IDs que estavam duplicados: ['C2388', 'C4576', 'C1456', 'C0807', 'C0497', 'C1268', 'C3601', 'C4374', 'C1151', 'C3976', 'C2163', 'C1114', 'C3386', 'C0857', 'C2033', 'C3618', 'C1643', 'C3711', 'C1235', 'C4645']


## Passo 3 — De-para

`config.DE_PARA` em `tipo_contrato` e `forma_pagamento`.

In [ ]:
# TODO

## Passo 4 — Sentinelas

`config.SENTINELAS` → NA (não imputar aqui).

In [ ]:
# TODO

## Passo 5 — Chamados

Contar datas entre aquisição e corte; vazio = 0.

In [ ]:
# TODO

## Passo 6 — Tempo de casa

Meses entre a aquisição e `config.DATA_CORTE` (datas: `format='mixed', dayfirst=True`).

In [ ]:
# TODO

## Passo 7 — Rótulo churn

1 se a data cai em (DATA_CORTE, FIM_JANELA]; 0 se vazia; remover churn antes da aquisição.

In [ ]:
# TODO

## Passo 8 — Maturação

Remover `tempo_cliente_meses < config.MATURACAO_MINIMA_MESES`.

In [ ]:
# TODO

## Passo 9 — Colunas finais e auditoria (depois)

Conferir: ~4.976 clientes, ~35,4% de churn.

In [ ]:
# TODO

## Baixar arquivos gerados

Depois, suba-os em `data/processed/` pelo GitHub (Add file → Upload files).

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(config.BASE_LIMPA))